# RedisFileSearchTool - Drop-in Search for Agents

This notebook demonstrates `RedisFileSearchTool` and the `create_redis_file_search_tool()` factory function.

## What is RedisFileSearchTool?

`RedisFileSearchTool` wraps a `HybridSearchService` (combined BM25 + vector similarity search) into a callable tool that integrates with the OpenAI Agents SDK. It provides:

- **Hybrid search** - Combines semantic (vector) and lexical (BM25) matching via Reciprocal Rank Fusion (RRF)
- **LLM-ready output** - Formats results as a readable string for LLM consumption
- **Parameter schema** - Exposes a JSON schema so the LLM knows how to call the tool
- **Score filtering** - Optional `min_score` to discard low-quality results
- **Customizable** - Name, description, and defaults can be configured per-agent

## 1. Setup and Document Indexing

First, create a `HybridSearchService` and index some sample documents.

In [ ]:
import os
import json

from redis_openai_agents import (
    HybridSearchService,
    RedisFileSearchTool,
    create_redis_file_search_tool,
)

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

# Create a HybridSearchService (BM25 + vector search with RRF fusion)
service = HybridSearchService(
    redis_url=REDIS_URL,
    index_name="file_search_demo",
    default_vector_weight=0.7,
    default_text_weight=0.3,
)

# Sample documents to index
documents = [
    {
        "content": "Redis is an open-source, in-memory data structure store used as a database, cache, and message broker. It supports data structures such as strings, hashes, lists, sets, and sorted sets.",
        "metadata": {"source": "overview.md", "category": "introduction"},
    },
    {
        "content": "Redis Search provides full-text search, secondary indexing, and vector similarity search capabilities. It supports hybrid queries combining text and vector search.",
        "metadata": {"source": "search.md", "category": "search"},
    },
    {
        "content": "Redis Streams is a log data structure that can be used for event sourcing, message queuing, and real-time data processing. Consumer groups allow multiple consumers to process stream entries.",
        "metadata": {"source": "streams.md", "category": "data-structures"},
    },
    {
        "content": "Redis JSON allows storing, updating, and retrieving JSON documents in Redis. It supports JSONPath queries for partial document access and atomic updates to nested fields.",
        "metadata": {"source": "json.md", "category": "data-structures"},
    },
    {
        "content": "Redis TimeSeries provides time series data storage with automatic downsampling, aggregation queries, and retention policies. It is ideal for metrics, IoT sensor data, and monitoring.",
        "metadata": {"source": "timeseries.md", "category": "data-structures"},
    },
    {
        "content": "Vector similarity search in Redis uses HNSW or FLAT indexing algorithms. HNSW provides approximate nearest neighbor search with sub-linear query time, while FLAT provides exact search.",
        "metadata": {"source": "vectors.md", "category": "search"},
    },
    {
        "content": "Redis persistence can be configured using RDB snapshots, AOF (Append Only File), or a combination of both. RDB creates point-in-time snapshots, while AOF logs every write operation.",
        "metadata": {"source": "persistence.md", "category": "operations"},
    },
    {
        "content": "Redis Cluster provides horizontal scaling by automatically partitioning data across multiple nodes. It uses hash slots for data distribution and supports automatic failover.",
        "metadata": {"source": "cluster.md", "category": "operations"},
    },
]

# Index documents in both vector and text stores
ids = await service.index_documents(documents)
count = await service.count()

print(f"Indexed {len(ids)} documents")
print(f"Total documents in index: {count}")

## 2. Create the Tool Using the Factory Function

The `create_redis_file_search_tool()` factory is the recommended way to create a search tool. It wraps the `HybridSearchService` and returns a `RedisFileSearchTool` instance.

In [ ]:
# Create the tool with default settings
search_tool = create_redis_file_search_tool(service)

print(f"Tool name:        {search_tool.name}")
print(f"Default k:        {search_tool.default_k}")
print(f"Default min_score: {search_tool.default_min_score}")
print(f"Type:             {type(search_tool).__name__}")
print(f"\nDescription:\n  {search_tool.description}")

## 3. Execute the Tool Directly

`RedisFileSearchTool` is callable -- you can invoke it directly with a query string. The tool returns a formatted string suitable for LLM consumption.

In [ ]:
# Call the tool with a search query
result = await search_tool(query="How does vector search work in Redis?")
print(result)

In [ ]:
# Try another query - keyword-oriented (will benefit from BM25)
result = await search_tool(query="persistence RDB AOF", k=3)
print(result)

In [ ]:
# Empty query returns a helpful message
result = await search_tool(query="")
print(result)

## 4. Customize Name, Description, and Defaults

The factory function accepts parameters to customize the tool for different agents or use cases.

In [ ]:
# Create a customized tool for a specific agent
docs_tool = create_redis_file_search_tool(
    service,
    name="search_redis_docs",
    description=(
        "Search the Redis documentation knowledge base. "
        "Use this to answer questions about Redis features, "
        "configuration, data structures, and operations."
    ),
    default_k=3,
    default_min_score=0.005,
)

print(f"Tool name:         {docs_tool.name}")
print(f"Default k:         {docs_tool.default_k}")
print(f"Default min_score: {docs_tool.default_min_score}")
print(f"\nDescription:\n  {docs_tool.description}")

# Execute the customized tool
print("\n" + "=" * 60)
result = await docs_tool(query="horizontal scaling and partitioning")
print(result)

## 5. Parameter Schema for LLM Function Calling

The tool exposes a `parameters` dict containing a JSON Schema. This schema tells the LLM what arguments the tool accepts, enabling proper function calling.

In [ ]:
# Inspect the parameter schema
print("Parameter schema (used by LLM for function calling):")
print("=" * 60)
print(json.dumps(search_tool.parameters, indent=2))

print("\n\nRequired parameters:", search_tool.parameters["required"])
print("\nProperties:")
for prop_name, prop_schema in search_tool.parameters["properties"].items():
    desc = prop_schema.get("description", "")
    ptype = prop_schema.get("type", "")
    default = prop_schema.get("default", "N/A")
    print(f"  - {prop_name} ({ptype}): {desc} [default: {default}]")

In [ ]:
# The customized tool has an updated schema reflecting its defaults
print("Customized tool schema (default_k=3):")
print("=" * 60)
print(json.dumps(docs_tool.parameters, indent=2))

## 6. Min-Score Filtering

The `min_score` parameter filters out results below a similarity threshold. This is useful for ensuring the agent only sees high-quality matches.

In [ ]:
# Search without min_score filtering (all results returned)
print("Without min_score filtering (all results):")
print("=" * 60)
result_all = await search_tool(query="time series metrics monitoring", k=5)
print(result_all)

In [ ]:
# Search with a min_score threshold to filter weak matches
print("With min_score=0.01 (only stronger matches):")
print("=" * 60)
result_filtered = await search_tool(
    query="time series metrics monitoring",
    k=5,
    min_score=0.01,
)
print(result_filtered)

In [ ]:
# With a very high min_score, fewer (or no) results pass the filter
print("With min_score=0.5 (very strict filtering):")
print("=" * 60)
result_strict = await search_tool(
    query="time series metrics monitoring",
    k=5,
    min_score=0.5,
)
print(result_strict)

## 7. Integration with an OpenAI Agent

The primary purpose of `RedisFileSearchTool` is to serve as a tool inside an OpenAI Agents SDK `Agent`. The tool's `name`, `description`, and `parameters` schema enable the LLM to discover and invoke the search when needed.

Below is a conceptual example showing how to wire the tool into an agent's tool list.

```python
from agents import Agent, Runner
from redis_openai_agents import HybridSearchService, create_redis_file_search_tool

# 1. Create the search service and index your documents
service = HybridSearchService(
    redis_url="redis://localhost:6379",
    index_name="support_kb",
)
await service.index_documents([
    {"content": "To reset your password, visit Settings > Security > Reset Password.",
     "metadata": {"source": "faq.md", "category": "account"}},
    {"content": "Billing invoices are available under Settings > Billing > History.",
     "metadata": {"source": "faq.md", "category": "billing"}},
    # ... more documents
])

# 2. Create the search tool
search_tool = create_redis_file_search_tool(
    service,
    name="search_support_kb",
    description="Search the customer support knowledge base for answers.",
    default_k=3,
    default_min_score=0.005,
)

# 3. Create the agent with the tool
support_agent = Agent(
    name="Support Agent",
    instructions=(
        "You are a customer support agent. Use the search_support_kb tool "
        "to find relevant answers before responding to the user."
    ),
    tools=[search_tool],
)

# 4. Run the agent
result = await Runner.run(support_agent, "How do I reset my password?")
print(result.final_output)
```

The agent will automatically call `search_support_kb` when it needs to look up information, passing the `query` (and optionally `k` and `min_score`) based on the parameter schema.

## 8. Direct Instantiation (Alternative)

You can also instantiate `RedisFileSearchTool` directly instead of using the factory function. This gives you access to all dataclass fields, including the ability to provide a custom `parameters` schema.

In [ ]:
# Direct instantiation
direct_tool = RedisFileSearchTool(
    _service=service,
    name="custom_search",
    description="A directly instantiated search tool.",
    default_k=10,
    default_min_score=0.0,
)

print(f"Tool name: {direct_tool.name}")
print(f"Default k: {direct_tool.default_k}")

# It works the same way
result = await direct_tool(query="message broker event sourcing")
print("\n" + result)

## Summary

`RedisFileSearchTool` and `create_redis_file_search_tool()` provide:

1. **Drop-in agent tool** - Wraps `HybridSearchService` into a callable tool for the OpenAI Agents SDK
2. **Hybrid search** - Combines BM25 (keyword) and vector (semantic) search with RRF score fusion
3. **LLM-friendly output** - Results are formatted as readable text with scores and metadata
4. **Function calling schema** - Exposes `parameters` JSON schema so the LLM can invoke the tool correctly
5. **Score filtering** - `min_score` removes low-quality results before they reach the LLM
6. **Customizable** - Name, description, default_k, and default_min_score are all configurable
7. **Error-safe** - Exceptions are caught and returned as strings rather than crashing the agent

In [ ]:
# Cleanup - delete all indexed documents
await service.delete_all()
print("Cleaned up: all indexed documents deleted.")